# 练习:在新数据集上独立完成多模态整合
### PBMC 10k CITE-seq(约 40 分钟,可在 notebook 中直接作答)

**前置要求**:已完成教程 Part II(多模态整合)。

**任务说明**:本练习提供一份**新的真实数据集**——另一位健康供体的 PBMC(10x Genomics,约 7,800 个细胞,同一模态:RNA + 表面蛋白)。请把教程中学到的完整流程**独立**应用到这份数据上,并回答每道思考题。

- 数据文件:`citeseq_pbmc10k.npz`(与本 notebook 同目录;预处理流程与教程数据一致,见 `prepare_data.py pbmc10k`)
- 带 `____` 标记的空缺代码需要你来补全;
- 每道题的参考答案折叠在"参考答案"中——**先独立完成,再展开核对**;
- 环境要求与教程一致(numpy / matplotlib / scikit-learn,CPU 即可)。

**评分点**:任务 1–4 的代码正确性(60%)+ 任务 5 思考题的分析质量(40%)。


In [ ]:
# ============================================================
#  SETUP -- 已为你准备好(与教程相同的工具函数)
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import CCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score, cross_val_predict

def standardize(X):
    return StandardScaler().fit_transform(X)

def show(emb, labels, title, ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(4.5, 4))
    for lab in np.unique(labels):
        m = labels == lab
        ax.scatter(emb[m, 0], emb[m, 1], s=15, label=str(lab))
    ax.set_title(title, fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])
    return ax

def score(emb, labels, name):
    s = silhouette_score(emb, labels)
    a = cross_val_score(KNeighborsClassifier(10), emb, labels, cv=5).mean()
    print(f'{name:28s} silhouette = {s: .3f}   kNN-acc = {a:.3f}')

print('Environment OK')

## 任务 1:认识数据(5 分钟)

运行下面的代码,回答:

1. 这份数据的稀疏性(零值比例)是多少?与教程的 5k 数据(94.5%)相比如何?
2. 这份数据的 ADT view 有多少种蛋白?(注意:与 5k 数据不同,这份数据的抗体 panel 更小——想想这对下游分析意味着什么。)

<details>
<summary><b>参考答案</b>(完成后再展开)</summary>

1. 约 94%,与 5k 数据相近——稀疏性是 scRNA-seq 数据的固有性质,与供体无关;
2. 14 种(5k 数据有 29 种)。蛋白 panel 更小意味着 ADT view 的信息量更少,标签规则可用的 marker 也更少(例如没有 CD20,只能靠 CD19 定义 B 细胞)——这会影响蛋白 view 单独分析的上限。
</details>


In [ ]:
# ---- 任务 1:加载数据并观察基本性质 ----
d = np.load('citeseq_pbmc10k.npz', allow_pickle=True)

print('RNA view:', d['X_rna'].shape, '   ADT view:', d['X_adt'].shape)
print('零值比例(RNA): %.1f%%' % ((d['X_rna'] == 0).mean() * 100))
print('细胞类型分布:', dict(zip(*np.unique(d['labels'], return_counts=True))))

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].hist(d['lib_size'], bins=50, color='#FF6C0C', alpha=0.85)
axes[0].set_title('Library size (total UMI per cell)')
axes[1].hist(d['n_genes'], bins=50, color='#0066cc', alpha=0.85)
axes[1].set_title('Detected genes per cell')
plt.tight_layout(); plt.show()

## 任务 2:质量控制(5 分钟)

真实数据分析的第一步是 QC。下面用与教程相同的分位数方法去除极端离群细胞。

1. 运行代码,记录去掉了多少个细胞;
2. 为什么要在分析前做这一步?(提示:离群细胞对 PCA 有什么影响?)

<details>
<summary><b>参考答案</b>(完成后再展开)</summary>

1. 约 140 个细胞(占 ~1.8%);
2. 少数极端离群细胞会把 PCA 的主方向拉向它们,使其余细胞被压缩到很小的区域,真实结构被掩盖。去除它们之后,主成分才能反映细胞群体的真实差异。
</details>


In [ ]:
# ---- 任务 2:QC(去除前后 0.5% 分位之外的离群细胞)----
_emb = PCA(2).fit_transform(standardize(d['X_rna']))
_lo, _hi = np.quantile(_emb, [0.005, 0.995], axis=0)
_keep = ((_emb >= _lo) & (_emb <= _hi)).all(axis=1)
print(f'QC: removed {(~_keep).sum()} outlier cells, kept {_keep.sum()} cells')

X_rna = d['X_rna'][_keep]
X_adt = d['X_adt'][_keep]
labels = d['labels'][_keep]
Z_rna = standardize(X_rna)
Z_adt = standardize(X_adt)

## 任务 3:单 view 与朴素拼接(10 分钟)

补全下面代码中的三处 `____`,完成三种 embedding 的对比:

- `emb_adt`:仅用 ADT view 的 PCA(2 维)
- `emb_rna`:仅用 RNA view 的 PCA(2 维)
- `emb_concat`:两个 view 标准化后拼接,再 PCA(2 维)

运行后回答:**在这份数据上,哪个 view 单独表现最好?拼接是否最好?**(用 silhouette 和 kNN-acc 两个指标回答)

<details>
<summary><b>参考答案</b>(完成后再展开)</summary>

补全的代码:

```python
emb_adt    = PCA(2).fit_transform(Z_adt)
emb_rna    = PCA(2).fit_transform(Z_rna)
emb_concat = PCA(2).fit_transform(np.hstack([Z_rna, Z_adt]))
```

参考结果(数值可能因软件版本略有波动):ADT only sil ≈ 0.28 / kNN ≈ 0.85;RNA only sil ≈ 0.37 / kNN ≈ 0.82;concat sil ≈ 0.38 / kNN ≈ 0.84。**2 维 silhouette 上 concat 最好;kNN 上 ADT 略高(标签来自蛋白 marker,蛋白 view 天然占优)——与教程 5k 数据的模式一致。**
</details>


In [ ]:
# ---- 任务 3:补全三处空缺 ----
emb_adt    = PCA(2).fit_transform(____)          # 只用 ADT view
emb_rna    = PCA(2).fit_transform(____)          # 只用 RNA view
emb_concat = PCA(2).fit_transform(np.hstack(____))  # 拼接两个 view

fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.2))
show(emb_adt,    labels, 'ADT only (14 proteins)',   ax=axes[0])
show(emb_rna,    labels, 'RNA only (1000 genes)',    ax=axes[1])
show(emb_concat, labels, 'Concat + PCA',             ax=axes[2])
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

score(emb_adt,    labels, 'ADT only')
score(emb_rna,    labels, 'RNA only')
score(emb_concat, labels, 'Concat + PCA')

## 任务 4:per-view PCA + CCA(10 分钟)

按照教程的标准流程补全代码:每个 view 先 PCA 降维(RNA 20 维、ADT 10 维),再运行 CCA(2 维)。回答:**CCA 在这份数据上超过 concat + PCA 了吗?为什么?**

<details>
<summary><b>参考答案</b>(完成后再展开)</summary>

补全的代码:

```python
P_rna = PCA(20).fit_transform(Z_rna)
P_adt = PCA(10).fit_transform(Z_adt)
U, _ = CCA(n_components=2).fit(P_rna, P_adt).transform(P_rna, P_adt)
```

参考结果:CCA 的 silhouette ≈ 0.34,kNN ≈ 0.85——**与 concat 相当,但没有超过**。原因与教程结论一致:这份数据的两个 view 高度一致、私有变异弱,CCA 没有用武之地;它的舞台是跨批次/跨平台整合。
</details>


In [ ]:
# ---- 任务 4:补全三处空缺 ----
P_rna = PCA(____).fit_transform(Z_rna)
P_adt = PCA(____).fit_transform(Z_adt)
U, _ = CCA(n_components=____).fit(P_rna, P_adt).transform(P_rna, P_adt)

fig, axes = plt.subplots(1, 2, figsize=(9, 4.2))
show(emb_concat, labels, 'Concat + PCA',       ax=axes[0])
show(U,          labels, 'Per-view PCA + CCA', ax=axes[1])
plt.tight_layout(); plt.show()

score(emb_concat, labels, 'Concat + PCA')
score(U,          labels, 'Per-view PCA + CCA')

## 任务 5:逐类准确率与思考题(10 分钟)

运行下面的逐类准确率表(已为你写好),然后回答三道思考题:


In [ ]:
# ---- 任务 5:逐类准确率(已提供,直接运行)----
P_cat = np.hstack([P_rna, P_adt])
emb_cca10, _ = CCA(n_components=10).fit(P_rna, P_adt).transform(P_rna, P_adt)

accs = {}
for nm, emb in [('RNA', P_rna), ('ADT', P_adt), ('Concat', P_cat), ('CCA', emb_cca10)]:
    pred = cross_val_predict(KNeighborsClassifier(10), emb, labels, cv=5)
    accs[nm] = {l: (pred[labels == l] == l).mean() for l in np.unique(labels)}

print(f'{"cell type":<10s}{"RNA only":>10s}{"ADT only":>10s}{"Concat":>10s}{"CCA":>8s}')
for l in np.unique(labels):
    print(f'{l:<10s}{accs["RNA"][l]:>10.2f}{accs["ADT"][l]:>10.2f}{accs["Concat"][l]:>10.2f}{accs["CCA"][l]:>8.2f}')

**思考题**:

1. 在教程的 5k 数据上,CD8 T 细胞仅靠 RNA view 的准确率只有约 0.46;在这份 10k 数据上却达到了约 0.85。**可能的原因有哪些?**(提示:从供体差异、抗体 panel 差异、标签规则差异三个角度思考)
2. `other` 类在四种方法下都难以预测(0.05–0.5)。为什么?(提示:这一类是怎么定义出来的?)
3. 综合任务 3–5,如果让你向一位同事推荐这份数据上的整合方案,你会推荐哪一种?用一句话说明理由。

<details>
<summary><b>参考答案</b>(完成后再展开)</summary>

1. 可能原因:(a) 供体间的生物学差异——不同个体的 CD8 T 细胞转录状态可能不同;(b) panel 差异——5k 有 29 种蛋白、10k 只有 14 种,标签规则随之不同(如 B 细胞在 10k 中只靠 CD19 定义),标签的"分辨率"变了;(c) 标签噪声——marker 规则是启发式的,阈值两侧的不确定性会以不同方式影响两个数据集。这说明:**跨数据集比较方法时,结论可能随数据和标签而变化——这也是基准测试(benchmark)要用多个数据集的原因。**
2. `other` 是由"所有 marker 得分都不高"的细胞拼凑而成的——它按定义就不是一个生物学上均一的群体(混有 DC、血小板、双联体、低质量细胞等),任何 embedding 里它都不会形成紧致的簇,自然无法被邻居投票准确预测。
3. 开放题。合理的答案:concat + per-view PCA(简单、稳健、效果与 CCA 相当);理由示例:"两个 view 一致性高,私有变异弱,简单拼接已足够;CCA 没有带来额外收益。"
</details>


## 挑战题(可选,+20%):把 Part I 用起来

这份 10k 数据和教程的 5k 数据来自**不同供体、不同实验批次**——它们正好可以当作单模态整合中的两个"批次"。

任务:

1. 分别加载 `citeseq_pbmc.npz`(5k)和 `citeseq_pbmc10k.npz`(10k)的 RNA view,取 `gene_names` 的交集(约 244 个基因),把两个矩阵按行合并;细胞类型标签体系一致,可以直接合并。**注意:标准化要在合并之后统一做**(如果先各自标准化,就等于已经做了逐批次中心化了——想想为什么);
2. 合并 + PCA,用 Part I 的两个指标(silhouette 按类型、batch mixing rate)评估;
3. 用 Part I 的**逐批次中心化**校正后重新评估。

你会发现两个"反常"的现象,请解释它们:

- 合并后不校正,mixing 本来就不算太低(> 0.4)——为什么这两个"批次"的差异没有模拟数据里那么夸张?
- 逐批次中心化之后,mixing 反而**变差**了——校正为什么会帮倒忙?(提示:比较两个供体的细胞类型**组成比例**;逐批次中心化减掉的"批次均值差"里,除了技术噪声还有什么?这正是**过度校正**的现实案例。)

<details>
<summary><b>参考要点</b>(完成后再展开)</summary>

- 两个数据集来自同一组织(PBMC)、同一平台(10x v3 + TotalSeq-B),实验流程高度一致,因此技术层面的批次效应远小于模拟数据中人为构造的强度;批次混合的"地板"本来就不低。
- 两个供体的细胞类型**组成不同**(例如 CD8 T 在 5k 中约占 10%、在 10k 中约占 18%)。逐批次中心化把每个批次的均值对齐,但批间均值差里既有技术噪声、也有**真实的组成差异**——一刀切地减掉,会把同类细胞在两个批次中推到不同的位置,反而降低混合度。这就是教程中"过度校正会抹掉真实生物学差异"在真实数据上的直接体现,也是实际分析中要用 Harmony/scVI 等更精细方法(而非简单中心化)的原因。
</details>
